
# HITO 2 – Análisis de Datos y Formulación del Problema
## Proyecto VPER

Índice:

1) Introducción
2) Objetivo de la exploración inicial
3) Análisis


## Introducción

Se analiza la estructura inicial del repositorio de datos utilizado en el proyecto Valorant Performance Evaluation Ranking (VPER), cuyo objetivo es desarrollar un sistema de inteligencia artificial capaz de evaluar el rendimiento individual de jugadores profesionales de Valorant.

Los datos no se encuentran consolidados en un único dataset listo para el análisis. En su lugar, el repositorio está organizado siguiendo una lógica similar a la de un data lake, donde la información se almacena de forma distribuida y segmentada por temporadas competitivas. Dentro de la carpeta principal /data existen distintas particiones anuales (vct_2021, vct_2022, ..., vct_2025) que contienen información específica de cada periodo, así como una carpeta global /all_ids que actúa como catálogo de identificadores del ecosistema competitivo.

Cada partición anual se divide a su vez en subcarpetas que representan diferentes tipos de información, como agents, ids, matches y player_stats, lo que permite separar las entidades principales del sistema competitivo. Esta organización facilita el almacenamiento histórico de los datos, aunque implica que el dataset final aún no está completamente consolidado y requiere procesos posteriores de integración y preparación antes de ser utilizado para modelado.

## Objetivo

El objetivo de esta exploración inicial es comprender la organización del data lake y el papel de cada uno de sus componentes antes de realizar el análisis de datos. En particular, se busca identificar la estructura del repositorio, los diferentes tipos de archivos disponibles y cómo se relacionan entre sí las entidades principales del sistema competitivo, como jugadores, equipos, partidas y torneos.

Esta etapa es fundamental para establecer una base sólida para el análisis exploratorio de datos (EDA), ya que permitirá determinar qué fuentes contienen la información relevante para el proyecto, cómo deben integrarse los distintos conjuntos de datos y cuál será la unidad de análisis más adecuada para evaluar el rendimiento competitivo de los jugadores.

## Análisis

### 1) Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### 2) Definir las rutas del Data Lake

In [2]:
DATA_PATH = "data/vct_2025"

agents_path = os.path.join(DATA_PATH, "agents")
ids_path = os.path.join(DATA_PATH, "ids")
matches_path = os.path.join(DATA_PATH, "matches")
player_stats_path = os.path.join(DATA_PATH, "players_stats")

### 3) Cargar datasets principales

In [3]:
# Player stats (dataset principal)
player_stats = pd.read_csv(os.path.join(player_stats_path, "players_stats.csv"))

# IDs
players_ids = pd.read_csv(os.path.join(ids_path, "players_ids.csv"))
teams_ids = pd.read_csv(os.path.join(ids_path, "teams_ids.csv"))

# Matches
maps_scores = pd.read_csv(os.path.join(matches_path, "maps_scores.csv"))
overview = pd.read_csv(os.path.join(matches_path, "overview.csv"))

# Ver tamaño
print("player_stats:", player_stats.shape)
print("players_ids:", players_ids.shape)
print("teams_ids:", teams_ids.shape)
print("maps_scores:", maps_scores.shape)
print("overview:", overview.shape)

player_stats: (17996, 25)
players_ids: (349, 2)
teams_ids: (58, 2)
maps_scores: (1277, 16)
overview: (53226, 21)


### 4) Inspección inicial

* Información general

In [4]:
player_stats.head()

,Tournament,Stage,Match Type,Player,Teams,Agents,Rounds Played,Rating,Average Combat Score,Kills:Deaths,"Kill, Assist, Trade, Survive %",Average Damage Per Round,Kills Per Round,Assists Per Round,First Kills Per Round,First Deaths Per Round,Headshot %,Clutch Success %,Clutches (won/played),Maximum Kills in a Single Map,Kills,Deaths,Assists,First Kills,First Deaths
0,Valorant Champions 2025,Playoffs,Upper Quarterfinals,Boo,Team Heretics,astra,26,0.68,129.0,0.61,69%,89.0,0.42,0.15,0.12,0.12,44%,NaN,0/4,11,11,18,4,3,3
1,Valorant Champions 2025,Playoffs,Upper Quarterfinals,Boo,Team Heretics,omen,19,0.46,98.0,0.40,42%,55.0,0.32,0.26,0.05,0.26,37%,33%,1/3,6,6,15,5,1,5
2,Valorant Champions 2025,Playoffs,Upper Quarterfinals,Boo,Team Heretics,"astra, omen",45,0.59,114.0,0.52,58%,74.0,0.38,0.20,0.09,0.18,41%,14%,1/7,11,17,33,9,4,8
3,Valorant Champions 2025,Playoffs,Upper Quarterfinals,RieNs,Team Heretics,fade,26,1.21,219.0,1.13,81%,147.0,0.69,0.38,0.08,0.04,23%,33%,1/3,18,18,16,10,2,1
4,Valorant Champions 2025,Playoffs,Upper Quarterfinals,RieNs,Team Heretics,sova,19,1.02,220.0,0.75,58%,142.0,0.63,0.26,0.05,0.05,28%,NaN,0/4,12,12,16,5,1,1


* Información detallada

In [5]:

player_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17996 entries, 0 to 17995
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Tournament                      17996 non-null  object 
 1   Stage                           17996 non-null  object 
 2   Match Type                      17996 non-null  object 
 3   Player                          17996 non-null  object 
 4   Teams                           17996 non-null  object 
 5   Agents                          17996 non-null  object 
 6   Rounds Played                   17996 non-null  int64  
 7   Rating                          16604 non-null  float64
 8   Average Combat Score            17986 non-null  float64
 9   Kills:Deaths                    17996 non-null  float64
 10  Kill, Assist, Trade, Survive %  16692 non-null  object 
 11  Average Damage Per Round        16626 non-null  float64
 12  Kills Per Round                 

* Estadísctica descriptiva

In [6]:
player_stats.describe()

,Rounds Played,Rating,Average Combat Score,Kills:Deaths,Average Damage Per Round,Kills Per Round,Assists Per Round,First Kills Per Round,First Deaths Per Round,Maximum Kills in a Single Map,Kills,Deaths,Assists,First Kills,First Deaths
count,17996.000000,16604.000000,17986.000000,17996.000000,16626.000000,17996.000000,17996.000000,16645.000000,16646.000000,17996.000000,17996.000000,17996.000000,17996.000000,17996.000000,17996.000000
mean,56.905979,0.988747,195.649283,1.036644,128.316312,0.688337,0.279657,0.099878,0.101812,16.917648,39.377195,39.402367,16.060736,5.273727,5.281118
std,73.473854,0.275065,49.065678,0.451347,30.629254,0.192783,0.147140,0.071383,0.068724,5.739903,52.716300,50.380248,23.288620,8.685712,8.120614
min,13.000000,0.030000,22.000000,0.000000,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,22.000000,0.820000,163.000000,0.770000,108.000000,0.560000,0.170000,0.050000,0.050000,13.000000,15.000000,15.000000,5.000000,1.000000,1.000000
50%,37.000000,0.980000,193.000000,0.980000,127.000000,0.680000,0.260000,0.090000,0.090000,17.000000,24.000000,24.000000,9.000000,3.000000,3.000000
75%,61.000000,1.150000,225.000000,1.210000,146.000000,0.800000,0.360000,0.140000,0.140000,21.000000,41.000000,41.000000,17.000000,6.000000,6.000000
max,696.000000,2.660000,509.000000,18.000000,307.000000,1.950000,1.570000,0.500000,0.530000,42.000000,579.000000,485.000000,345.000000,137.000000,111.000000


### 5) Verificar valores nulos

In [7]:
player_stats.isnull().sum().sort_values(ascending=False)

Clutch Success %                  11337
Clutches (won/played)              4733
Rating                             1392
Average Damage Per Round           1370
First Kills Per Round              1351
First Deaths Per Round             1350
Headshot %                         1308
Kill, Assist, Trade, Survive %     1304
Average Combat Score                 10
Agents                                0
Rounds Played                         0
Player                                0
Teams                                 0
Match Type                            0
Stage                                 0
Tournament                            0
Kills:Deaths                          0
Assists Per Round                     0
Kills Per Round                       0
Maximum Kills in a Single Map         0
Kills                                 0
Deaths                                0
Assists                               0
First Kills                           0
First Deaths                          0


### 6) Visualizar valores duplicados

In [8]:
player_stats.duplicated().sum()

np.int64(0)

### 7) Identificar columnas a trabajar

In [9]:
player_stats.columns

Index(['Tournament', 'Stage', 'Match Type', 'Player', 'Teams', 'Agents',
       'Rounds Played', 'Rating', 'Average Combat Score', 'Kills:Deaths',
       'Kill, Assist, Trade, Survive %', 'Average Damage Per Round',
       'Kills Per Round', 'Assists Per Round', 'First Kills Per Round',
       'First Deaths Per Round', 'Headshot %', 'Clutch Success %',
       'Clutches (won/played)', 'Maximum Kills in a Single Map', 'Kills',
       'Deaths', 'Assists', 'First Kills', 'First Deaths'],
      dtype='object')

### 8) Limpieza de columnas innecesarias

> ⚠️ **NOTE:** Discutir si vamos a trabajar con estas columnas o añadir mas información como los mapas y las armas

In [10]:
columns_to_keep = [
    "Player",
    "Agents",
    "Kills",
    "Deaths",
    "Assists",
    "Average Damage Per Round",
    "Assists Per Round",
    "Kills Per Round",
    "First Kills",
    "First Deaths"
]

player_stats_clean = player_stats[columns_to_keep]

### 9) Verificar Dataset

In [11]:
player_stats_clean.head()

,Player,Agents,Kills,Deaths,Assists,Average Damage Per Round,Assists Per Round,Kills Per Round,First Kills,First Deaths
0,Boo,astra,11,18,4,89.0,0.15,0.42,3,3
1,Boo,omen,6,15,5,55.0,0.26,0.32,1,5
2,Boo,"astra, omen",17,33,9,74.0,0.20,0.38,4,8
3,RieNs,fade,18,16,10,147.0,0.38,0.69,2,1
4,RieNs,sova,12,16,5,142.0,0.26,0.63,1,1


In [12]:
player_stats_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17996 entries, 0 to 17995
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Player                    17996 non-null  object 
 1   Agents                    17996 non-null  object 
 2   Kills                     17996 non-null  int64  
 3   Deaths                    17996 non-null  int64  
 4   Assists                   17996 non-null  int64  
 5   Average Damage Per Round  16626 non-null  float64
 6   Assists Per Round         17996 non-null  float64
 7   Kills Per Round           17996 non-null  float64
 8   First Kills               17996 non-null  int64  
 9   First Deaths              17996 non-null  int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 1.4+ MB


### 10) Guardar el Dataset

In [13]:
player_stats_clean.to_csv("dataset_clean_vct.csv", index=False)